**The required libraries are imported here**

In [ ]:
from pathlib import Path


import pandas as pd
import numpy as np

from five_safes_tes_workbench.workbench import Workbench
from partialstats.partials import SumOfSquaresPartial
from partialstats.combiners import mean_combiner, variance_combiner
from partialstats.combiners.scalar import sum_combiner

**Validate workbench configuration**

In [ ]:
wb = Workbench()

wb.validate(config_path="config.yml") #type: ignore

 **Define SQL query**

The query compare systolic blood pressure between two groups:

- people with primary malignant neoplasm of skin
- people without primary malignant neoplasm of skin

The two OMOP concept IDs used are:

- `3004249`: systolic blood pressure
- `139750`: primary malignant neoplasm of skin

In [ ]:
sys_pressure_neoplasm_query = """
WITH last_occurrence AS (
    SELECT
        person_id,
        value_as_number,
        ROW_NUMBER() OVER (
            PARTITION BY person_id
            ORDER BY measurement_datetime DESC NULLS LAST
        ) AS rn
    FROM "DelphiDemo".measurement
    WHERE measurement_concept_id = 3004249
      AND value_as_number IS NOT NULL
),

value_with_status AS (
    SELECT
        value_as_number,
        CASE
            WHEN person_id IN (
                SELECT person_id
                FROM "DelphiDemo".condition_occurrence
                WHERE condition_concept_id = 139750
            )
            THEN 'with'
            ELSE 'without'
        END AS condition_status
    FROM last_occurrence
    WHERE rn = 1
)

SELECT
    condition_status,
    COUNT(value_as_number) AS count,
    SUM(value_as_number) AS sum,
    SUM(value_as_number * value_as_number) AS sumsq
FROM value_with_status
GROUP BY condition_status;
"""

wb.build_tes.simple_sql(
    name="Mean and variance systolic blood pressure",
    query=sys_pressure_neoplasm_query
)

wb.submit()

**Fetch approved outputs**

- After the task has completed and the outputs have been approved for egress.

In [ ]:
paths = wb.fetch_outputs()
contingency_paths = [v[0] for k, v in paths.items()]

**Collect partial statistics from each TRE output**

This function reads one output CSV file and extracts the row for a selected group, such as `with` or `without`.

Each TRE output contains the partial statistics needed to calculate mean and variance:

- `count`: number of records in the group
- `sum`: total systolic blood pressure for the group
- `sumsq`: sum of squared systolic blood pressure values for the group

These values are stored in a `SumOfSquaresPartial` object so they can be combined across TREs.

In [ ]:
def collect_var_data(
    path: Path,
    group_var: str = "condition_status",
    group_level: str = "with"
) -> SumOfSquaresPartial:
    data = pd.read_csv(path)

    row_data = data[data[group_var] == group_level]

    return SumOfSquaresPartial(
        count=row_data["count"].iloc[0],
        sum=row_data["sum"].iloc[0],
        sumsq=row_data["sumsq"].iloc[0],
    )

**Combine partial statistics and calculate mean and variance**

For each condition group, read the partial result from each TRE and performs mean and variance analysis. 

The final result is stored in `summary_table` and prints it.

In [ ]:
group_levels = ["with", "without"]

summary_rows = []

for group in group_levels:
    group_partials = []

    for path in data_paths:
        partial = collect_var_data(path, group_level=group) #type: ignore
        group_partials.append(partial)

    count = sum_combiner.combine(group_partials)
    mean = mean_combiner.combine(group_partials)
    variance = variance_combiner.combine(group_partials)
    standard_deviation = np.sqrt(variance)

    summary_rows.append({
        "condition_status": group,
        "count": count,
        "mean_systolic_blood_pressure": mean,
        "variance_systolic_blood_pressure": variance,
        "standard_deviation": standard_deviation,
    })

summary_table = pd.DataFrame(summary_rows)

summary_table